# GraphSAGE M2 귀속 + 사용자 CLV 행 가중치 M5 (Dunnhumby, seed 42)

기존 GraphSAGE M2는 그대로 유지하고, binary user-degree 10분위 안에서 `q_C`·`q_V`를 공동 순열한 M2 대조군으로 올바른 CLV 배정의 효과를 확인합니다. M4는 상품 가격·가격구간 적합도를 제거하고 `1 + 0.5*q_C(u)`로 사용자의 학습행만 가중합니다.

`DAY 1~683` 학습, `DAY 684~690` 신규상품 개발평가이며 final test·holdout은 만들지 않습니다. 완료 실험 `2d01d43c6d43`의 M1·M2를 검증 후 재사용하고, Drive에 파일이 없으면 오류로 멈추지 않고 M1·M2도 재학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys
PINNED_SOURCE_COMMIT = '7973e3e22e31378747647f8b33127e38ac845b0d'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
repo = Path('/content/clv-m2-lightgcn-runner')
errors = []
for attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists(): shutil.rmtree(repo)
    result = subprocess.run(['git','clone',REPO_URL,str(repo)], text=True, capture_output=True)
    if result.returncode == 0: break
    errors.append(result.stderr.strip()); print(f'GitHub clone {attempt}/3 실패:', result.stderr.strip())
else: raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(errors))
subprocess.run(['git','-C',str(repo),'checkout','-q',PINNED_SOURCE_COMMIT], check=True)
actual_sha = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip()
assert actual_sha == PINNED_SOURCE_COMMIT
for name in tuple(sys.modules):
    if name.startswith(('graphsage_','gnn_clv_','clv_scaled_','lightgcn_','clv_')): del sys.modules[name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from graphsage_clv_m2_user_clv_weight_screen import (
    configure_graphsage_m2_user_clv_weight_screen,
    preflight_summary,
    run_graphsage_m2_user_clv_weight_screen,
)
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_graphsage_m2_user_clv_weight_screen()
summary = preflight_summary(cfg)
assert summary['fixed']['new_item_task']
assert summary['fixed']['train_pairs_excluded_from_evaluation']
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert not summary['fixed']['final_test_constructed']
assert not summary['fixed']['holdout_constructed']
assert summary['fixed']['one_training_loop_and_optimizer_per_arm']
assert not summary['m2_unchanged']['external_reranking']
assert not summary['modified_m4']['item_price_or_bin_fit_in_weight']
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_graphsage_m2_user_clv_weight_screen(cfg)

In [ ]:
from IPython.display import display
def show(frame):
    view = frame.copy(); view.attrs = {}; display(view)
core = ['model_id','role','training_origin','recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10','coverage@10']
print('1) GraphSAGE M1·M2·M2 순열·수정 M4·수정 M5 핵심 절대지표'); show(result_df[[c for c in core if c in result_df.columns]])
print('2) M1·M2·수정 M4 대비 전체 지표 비교'); show(result_df.attrs['comparison'])
print('3) 수정 M4 기준 2x2 요인효과와 상호작용'); show(result_df.attrs['interaction'])
print('4) 새로 학습한 M2 경로의 입력 on/off 진단'); show(result_df.attrs['counterfactual_m2']); show(result_df.attrs['top10_overlap'])
print('5) 사전 고정 판독'); print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('6) 저장 파일'); print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))